In [1]:
import sys
from pathlib import Path

import pandas as pd

# Raíz del proyecto (para poder importar src/ desde notebooks/)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.auxiliar_functions import RAW_DATA_DIR, ARCHIVOS_CRUDOS

print("Raíz del proyecto:", PROJECT_ROOT)
print("Archivos:", ARCHIVOS_CRUDOS)

Raíz del proyecto: c:\Users\genat\proyectos\tp-final-datos-ii
Archivos: ['Totalizadores Planta 2020_2022.xlsx', 'Totalizadores Planta - 2021_2023.xlsx', 'Totalizadores Planta 2022_2023.xlsx']


In [2]:
import pandas as pd
from src.auxiliar_functions import RAW_DATA_DIR, ARCHIVOS_CRUDOS

for hoja in ["Consolidado Produccion", "Totalizadores Produccion"]:
    print(f"\n===== {hoja} =====")
    columnas = {}
    for archivo in ARCHIVOS_CRUDOS:
        df = pd.read_excel(RAW_DATA_DIR / archivo, sheet_name=hoja)
        columnas[archivo] = set(df.columns)
        print(f"{archivo}: {df.shape[0]} filas, {df.shape[1]} columnas")

    comunes = set.intersection(*columnas.values())
    print("Columnas en los 3 archivos:", len(comunes))
    for archivo, cols in columnas.items():
        solo_aca = cols - comunes
        if solo_aca:
            print(f"  Solo en {archivo}: {sorted(solo_aca)}")


===== Consolidado Produccion =====
Totalizadores Planta 2020_2022.xlsx: 15573 filas, 12 columnas
Totalizadores Planta - 2021_2023.xlsx: 15317 filas, 13 columnas
Totalizadores Planta 2022_2023.xlsx: 12010 filas, 18 columnas
Columnas en los 3 archivos: 12
  Solo en Totalizadores Planta - 2021_2023.xlsx: ['Hl de Mosto Copia']
  Solo en Totalizadores Planta 2022_2023.xlsx: ['Hl de Mosto Copia', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18']

===== Totalizadores Produccion =====
Totalizadores Planta 2020_2022.xlsx: 15573 filas, 40 columnas
Totalizadores Planta - 2021_2023.xlsx: 15316 filas, 40 columnas
Totalizadores Planta 2022_2023.xlsx: 12009 filas, 41 columnas
Columnas en los 3 archivos: 39
  Solo en Totalizadores Planta 2020_2022.xlsx: ['Hl Reserva 7']
  Solo en Totalizadores Planta - 2021_2023.xlsx: ['Hl Reserva 7']
  Solo en Totalizadores Planta 2022_2023.xlsx: ['Hl Session IPA', 'Id']


Consolidado Produccion: hay 12 columnas comunes a los 3 archivos (DIA, HORA y 10 de producción). Lo extra es basura: Hl de Mosto Copia es una copia y las Unnamed son columnas vacías del Excel. Trabajamos con las 12 comunes.

Totalizadores Produccion: Hl Reserva 7 desaparece y aparece Hl Session IPA. Las dos están en el mismo lugar de la lista. Lo más probable es que sea la misma columna renombrada: un casillero “reserva” que después se usó para una cerveza nueva esto es una hipotesis, lo confirmamos viendo si en el solapamiento entre archivos tienen los mismos valores. Id es un número de fila, basura.

In [5]:
# Cierre diario (23:59) de una hoja, unificando los 3 archivos con la misma regla que en EE:
# - dentro de cada archivo, el primer cierre de cada día (por las jornadas apiladas)
# - entre archivos, gana el más reciente
def cierre_diario(hoja):
    partes = []
    for prioridad, archivo in enumerate(ARCHIVOS_CRUDOS):  # 0 = más viejo, 2 = más reciente
        df = pd.read_excel(RAW_DATA_DIR / archivo, sheet_name=hoja)
        df = df[df["HORA"].astype(str).str.startswith("23:59")]
        df = df.drop_duplicates(subset="DIA", keep="first").copy()
        df["prioridad"] = prioridad
        # Convertir a número todas las columnas de medición (un ' ' suelto vuelve texto la columna entera)
        columnas_medicion = [c for c in df.columns if c not in ["DIA", "HORA"]]
        df[columnas_medicion] = df[columnas_medicion].apply(pd.to_numeric, errors="coerce")
        partes.append(df)
    todo = pd.concat(partes).sort_values(["DIA", "prioridad"])
    return todo.drop_duplicates(subset="DIA", keep="last").drop(columns="prioridad").reset_index(drop=True)

df_prod_diario = cierre_diario("Consolidado Produccion")
df_tot_prod_diario = cierre_diario("Totalizadores Produccion")
print("Consolidado Produccion:", df_prod_diario.shape, "| Totalizadores Produccion:", df_tot_prod_diario.shape)

# Columnas numéricas: cuántos días con cierre en 0 y estadísticas básicas
for nombre, df in [("Consolidado Produccion", df_prod_diario), ("Totalizadores Produccion", df_tot_prod_diario)]:
    numericas = df.select_dtypes("number")
    resumen = pd.DataFrame({
        "dias_en_cero_%": (numericas == 0).mean().mul(100).round(1),
        "media": numericas.mean().round(1),
        "max": numericas.max(),
        "nulos": numericas.isna().sum(),
    })
    print(f"\n===== {nombre} =====")
    print(resumen.sort_values("dias_en_cero_%").to_string())

Consolidado Produccion: (1185, 18) | Totalizadores Produccion: (1185, 42)

===== Consolidado Produccion =====
                     dias_en_cero_%   media      max  nulos
Unnamed: 14                     0.0     NaN      NaN   1185
Unnamed: 15                     0.8  2782.9   9193.0   1143
Unnamed: 16                     0.8  1468.1   4537.0   1143
Unnamed: 17                     0.9  1182.6   3287.0   1143
Hl Producido Bodega             2.3  8546.8  15315.5      0
Unnamed: 18                     2.5   285.5   3139.0   1143
Hl Cerveza Filtrada             4.6  7947.3  15883.0      0
Hl Cerveza Envasada             5.8  8483.1  15989.0      0
Cocimientos Diarios             8.4    10.3     24.0    252
Hl de Mosto Copia               9.8  5098.6  10303.0    358
Hl Cerveza Cocina              11.6  8610.6  16318.8      0
Hl de Mosto                    11.6  5474.4  10303.0      0
Hl Cerveza L4                  15.4  4058.5   6544.0      0
Hl Cerveza L2                  17.6  1366.5   4136

las Unnamed: 14 a 18, Hl de Mosto Copia e Id son columnas basura, las descartamos

Unnamed: 15 a 18 tienen números en 42 días, son columnas sin nombre en el Excel de 2022_2023 que seguramente alguien utilizo para cálculos sueltos al costado de la tabla


Hl Reserva 7 y Hl Session IPA no importan: valen 0 siempre y la hipótesis de que es la misma columna renombrada da igual, descartamos las dos.

con las marcas de cervezas, 9 están en 0 el 100 % de los días y otras 10 tienen más del 98 % de días en 0. Se descartan. Solo Tecate (39 % de días en 0) y algo Heineken (73 %) tienen producción regular. Además, el total de mosto ya está en Hl de Mosto de la otra hoja, así que el detalle por marca aporta poco

Hl Cerveza L2 a L5 y Cocimientos Diarios están en las dos hojas, con estadísticas idénticas. Se usan una sola vez, desde Consolidado Produccion.

Nivel Silo Bagazo Sur (2) y Nivel Tk Restos Lev no son acumulados (máximo de ~95 a 100, son porcentajes de llenado) y tienen muchos nulos. Nivel Silo Bagazo Norte (1) no aparece en la tabla. Como select_dtypes("number") solo toma columnas numéricas, debe estar como texto en algún archivo.



In [6]:
# Frio diario desde Consolidado EE con la misma función (exploratorio: sin limpiar inválidos)
df_ee_diario = cierre_diario("Consolidado EE")[["DIA", "Frio (Kw)"]]
df_ee_diario["Frio (Kw)"] = pd.to_numeric(df_ee_diario["Frio (Kw)"], errors="coerce")
df_ee_diario = df_ee_diario[df_ee_diario["Frio (Kw)"].between(0, 100_000)]

candidatas = ["Hl de Mosto", "Hl Cerveza Cocina", "Hl Producido Bodega", "Hl Cerveza Filtrada",
              "Hl Cerveza Envasada", "Hl Cerveza L2", "Hl Cerveza L3", "Hl Cerveza L4", "Hl Cerveza L5",
              "Cocimientos Diarios"]

# Unimos por fecha y reindexamos al calendario para que shift(-1) sea el día siguiente
df_cruce = df_prod_diario[["DIA"] + candidatas].merge(df_ee_diario, on="DIA").set_index("DIA").asfreq("D")
df_cruce["Frio_manana"] = df_cruce["Frio (Kw)"].shift(-1)

correlaciones = pd.DataFrame({
    "r_con_frio_hoy": df_cruce[candidatas].corrwith(df_cruce["Frio (Kw)"]),
    "r_con_frio_manana": df_cruce[candidatas].corrwith(df_cruce["Frio_manana"]),
}).round(3).sort_values("r_con_frio_manana", ascending=False)
print(correlaciones)

                     r_con_frio_hoy  r_con_frio_manana
Hl Producido Bodega           0.620              0.551
Hl Cerveza Envasada           0.503              0.478
Hl Cerveza Filtrada           0.584              0.448
Hl Cerveza L4                 0.446              0.427
Hl de Mosto                   0.482              0.394
Hl Cerveza Cocina             0.474              0.393
Cocimientos Diarios           0.462              0.385
Hl Cerveza L3                 0.357              0.347
Hl Cerveza L2                 0.303              0.271
Hl Cerveza L5                 0.240              0.206


La mejor es Hl Producido Bodega: r = 0,55, seguida de Envasada (0,48), Filtrada (0,45) y L4 (0,43). Debemos tener en cuenta la estacionalidad, ya que ne verano se produce y se vende mas cerveza, y tambien hace mas calos. Parte de esta correlacion puede venir de que las dos variables dependen de la estacion, no de que producir enfrie mas. 

In [14]:
# Hoja horaria unificada: misma regla que en EE (primeras 25 filas por día y gana el archivo más reciente)
def horario_unificado(hoja, columnas):
    partes = []
    for prioridad, archivo in enumerate(ARCHIVOS_CRUDOS):
        df = pd.read_excel(RAW_DATA_DIR / archivo, sheet_name=hoja)
        df = df.reindex(columns=["DIA", "HORA"] + columnas)  # si una columna no existe en este archivo, queda NaN
        df[columnas] = df[columnas].apply(pd.to_numeric, errors="coerce")
        df = df.dropna(subset=["DIA"]).groupby("DIA").head(25)  # jornadas apiladas
        df["prioridad"] = prioridad
        partes.append(df)
    todo = pd.concat(partes)
    # Por cada día, nos quedamos con las filas del archivo de mayor prioridad
    prioridad_max = todo.groupby("DIA")["prioridad"].transform("max")
    return todo[todo["prioridad"] == prioridad_max].drop(columns="prioridad").reset_index(drop=True)

columnas_prod = ["Hl de Mosto", "Hl Cerveza Cocina", "Hl Producido Bodega", "Hl Cerveza Filtrada",
                 "Hl Cerveza Envasada", "Hl Cerveza L2", "Hl Cerveza L3", "Hl Cerveza L4", "Hl Cerveza L5",
                 "Cocimientos Diarios"]
df_prod_horario = horario_unificado("Consolidado Produccion", columnas_prod)
print(df_prod_horario.shape, "| días:", df_prod_horario["DIA"].nunique())

(29698, 12) | días: 1190


In [9]:
hora_txt = df_prod_horario["HORA"].astype(str)

# a) Horas "mal escritas": que no terminan en :00:00 (salvo el cierre 23:59)
horas_raras = hora_txt[~hora_txt.str.endswith(":00:00") & ~hora_txt.str.startswith("23:59")]
print("a) Horas con segundos o formato raro:", len(horas_raras), "| ejemplos:", horas_raras.unique()[:8])

# b) Días incompletos (menos de 24 lecturas)
lecturas = df_prod_horario.groupby("DIA").size()
print("b) Días con < 24 lecturas:", (lecturas < 24).sum())

# b2) Días donde se repite una hora (comparando HH:MM, sin los segundos)
df_prod_horario["HH_MM"] = hora_txt.str[:5]
repetidas = df_prod_horario[df_prod_horario.duplicated(subset=["DIA", "HH_MM"], keep=False)]
dias_con_repetidas = repetidas["DIA"].nunique()
print(f"b2) Días con alguna hora repetida: {dias_con_repetidas} ({len(repetidas)} filas involucradas)")
if dias_con_repetidas > 0:
    print(repetidas[["DIA", "HORA"] + columnas_prod[:3]].head(10).to_string(index=False))

# c) a e) Por columna: sin reinicio a las 00:00, acumulado que baja, valores imposibles, saltos anómalos
df_ord = df_prod_horario.sort_values(["DIA", "HORA"])
filas = []
for col in columnas_prod:
    incremento = df_ord.groupby("DIA")[col].diff()
    medianoche = df_ord[hora_txt.loc[df_ord.index].str.startswith("00:00")][col]
    p999 = incremento[incremento > 0].quantile(0.999)
    filas.append({
        "columna": col,
        "dias_sin_reinicio": (medianoche > 0).sum(),
        "horas_que_bajan": (incremento < 0).sum(),
        "valores_negativos": (df_ord[col] < 0).sum(),
        "valores_>=1e6": (df_ord[col] >= 1_000_000).sum(),
        "p99.9_incremento": round(p999, 1),
        "nulos": df_ord[col].isna().sum(),
    })
print("\nc) Controles por columna:")
print(pd.DataFrame(filas).set_index("columna").to_string())

a) Horas con segundos o formato raro: 420 | ejemplos: <StringArray>
['20:00:01', '22:00:01', '12:00:01', '15:00:01', '03:00:01', '02:00:01',
 '10:00:01', '13:00:01']
Length: 8, dtype: str
b) Días con < 24 lecturas: 8
b2) Días con alguna hora repetida: 3 (6 filas involucradas)
       DIA     HORA  Hl de Mosto  Hl Cerveza Cocina  Hl Producido Bodega
2020-09-05 15:00:00       5076.0             7646.7              7395.35
2020-09-05 15:00:00       5452.0             8323.5              8014.25
2022-12-10 16:00:00          0.0                0.0               529.50
2022-12-10 16:00:00          0.0                0.0               529.50
2023-02-28 00:00:00          0.0                0.0                 0.00
2023-02-28 00:00:00          0.0                0.0                 0.00

c) Controles por columna:
                     dias_sin_reinicio  horas_que_bajan  valores_negativos  valores_>=1e6  p99.9_incremento  nulos
columna                                                               

Las 420 horas con :01 no son un problema, porque siempre se filtra con startswith o comparando HH:MM

los dias incompletos son los mismos 8 que en Consolidado EE, con las mismas fechas exactas. Tiene sentido, porque todas las hojas salen del mismo registrador.

hay 3 horas repetidas, que no rompen nada pero las modificaremos, 2020-09-05 15:00 aparece dos veces y falta 16:00, y el segundo valor es mayor que el primero, por lo que es la lectura de las 16:00.
2022-12-10 el orden de las horas es 13, 16, 15, 16: la de las 14:00 está etiquetada como 16:00.
2023-02-28 ya esta contemplada en la 1.1

Las columnas de producción están limpias. Sin valores imposibles, sin negativos, siempre reinician a las 00:00 y nunca bajan.

Cocimiento Diarios tiene 70% de sus valores nulos en el archivo 2020_2022, ya que en este periodo no se registraba, y en 52 dias no reinicia o no baja. Tambien tiene 252 dias sin datos y si informacion esta ya en Hl de Mosto, ya que los cocimientos son los lotes de mosto, por  lo que la descartamos.
 



In [10]:
# Horas que debería tener un día completo, en orden
HORAS_ESPERADAS = [f"{h:02d}:00" for h in range(24)] + ["23:59"]

def normalizar_horas(df):
    """Corrige las etiquetas de HORA sin tocar los valores medidos.
    Supone que las filas de cada día están en el orden del Excel (no reordenar por HORA antes)."""
    df = df.copy()
    df["HORA_ORIGINAL"] = df["HORA"].astype(str)

    # 1) Quitar los segundos: '04:00:01' -> '04:00'
    df["HORA"] = df["HORA_ORIGINAL"].str[:5]

    # 2) Un '00:00' que no es la primera fila del día es el inicio de otra jornada: se descarta
    posicion = df.groupby("DIA").cumcount()
    cero_intruso = (df["HORA"] == "00:00") & (posicion > 0)
    df = df[~cero_intruso].copy()

    # 3) En los días con las 25 filas, la posición indica la hora: corregimos las etiquetas que no coinciden
    posicion = df.groupby("DIA").cumcount()
    filas_del_dia = df.groupby("DIA")["HORA"].transform("size")
    hora_esperada = posicion.map(lambda i: HORAS_ESPERADAS[i] if i < 25 else None)
    mal_etiquetada = (filas_del_dia == 25) & (df["HORA"] != hora_esperada)
    df.loc[mal_etiquetada, "HORA"] = hora_esperada[mal_etiquetada]

    print(f"Filas '00:00' intrusas eliminadas: {cero_intruso.sum()}")
    print(f"Horas mal etiquetadas corregidas: {mal_etiquetada.sum()}")
    return df

df_prod_horario_ok = normalizar_horas(df_prod_horario)

Filas '00:00' intrusas eliminadas: 1
Horas mal etiquetadas corregidas: 2


In [11]:
# 1) Ya no debería haber horas repetidas
repetidas = df_prod_horario_ok.duplicated(subset=["DIA", "HORA"]).sum()
print("Horas repetidas después del arreglo:", repetidas)

# 2) Las filas corregidas: etiqueta original vs nueva
cambiadas = df_prod_horario_ok[df_prod_horario_ok["HORA_ORIGINAL"].str[:5] != df_prod_horario_ok["HORA"]]
print(cambiadas[["DIA", "HORA_ORIGINAL", "HORA", "Hl de Mosto", "Hl Producido Bodega"]].to_string(index=False))

# 3) En los días corregidos el acumulado tiene que seguir subiendo (nunca bajar)
dias_corregidos = cambiadas["DIA"].unique()
tramo = df_prod_horario_ok[df_prod_horario_ok["DIA"].isin(dias_corregidos)].sort_values(["DIA", "HORA"])
bajadas = (tramo.groupby("DIA")["Hl Producido Bodega"].diff() < 0).sum()
print("\nHoras que bajan en los días corregidos (debería ser 0):", bajadas)

Horas repetidas después del arreglo: 0
       DIA HORA_ORIGINAL  HORA  Hl de Mosto  Hl Producido Bodega
2020-09-05      15:00:00 16:00       5452.0              8014.25
2022-12-10      16:00:00 14:00          0.0               529.50

Horas que bajan en los días corregidos (debería ser 0): 0


In [12]:
diferencia = df_prod_diario["Hl Cerveza Envasada"] - df_prod_diario[["Hl Cerveza L2", "Hl Cerveza L3", "Hl Cerveza L4", "Hl Cerveza L5"]].sum(axis=1)
print(diferencia.describe())

count    1185.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
dtype: float64


vemos algo bueno para lo que sigue, una de las candidatas Hl cerveza envasada, es la suma de las 4 lineas de envasado. Si metemos las 5 en un modelo es informacion repetida. Lo mismo pasa, en menor medida, con Bodega, Filtrada y Envasada, que son etapas consecutivas del mismo proceso: la cerveza que se produce en bodega después se filtra y se envasa. Deberían estar muy correlacionadas entre sí.

esto lo tendremos en cuenta mas adelante

Lista de candidatas: "Hl de Mosto", "Hl Cerveza Cocina", "Hl Producido Bodega", "Hl Cerveza Filtrada",
                 "Hl Cerveza Envasada", "Hl Cerveza L2", "Hl Cerveza L3", "Hl Cerveza L4", "Hl Cerveza L5"
                 

In [13]:
def controles_calidad(df_horario, columnas):
    """Controles por columna sobre una hoja horaria (ya normalizada con normalizar_horas)."""
    df_ord = df_horario.sort_values(["DIA", "HORA"])
    filas = []
    for col in columnas:
        incremento = df_ord.groupby("DIA")[col].diff()
        medianoche = df_ord.loc[df_ord["HORA"] == "00:00", col]
        filas.append({
            "columna": col,
            "nulos_%": round(df_ord[col].isna().mean() * 100, 1),
            "dias_sin_reinicio": (medianoche > 0).sum(),
            "horas_que_bajan": (incremento < 0).sum(),
            "negativos": (df_ord[col] < 0).sum(),
            ">=1e6": (df_ord[col] >= 1_000_000).sum(),
            "p99.9_incremento": round(incremento[incremento > 0].quantile(0.999), 1),
        })
    return pd.DataFrame(filas).set_index("columna")


def correlacion_con_frio(df_diario_hoja, columnas):
    """Correlación del cierre diario de cada columna con Frio de hoy y de mañana."""
    cruce = df_diario_hoja[["DIA"] + columnas].merge(df_ee_diario, on="DIA").set_index("DIA").asfreq("D")
    cruce["Frio_manana"] = cruce["Frio (Kw)"].shift(-1)
    return pd.DataFrame({
        "r_con_frio_hoy": cruce[columnas].corrwith(cruce["Frio (Kw)"]),
        "r_con_frio_manana": cruce[columnas].corrwith(cruce["Frio_manana"]),
        "dias_en_cero_%": (cruce[columnas] == 0).mean().mul(100),
    }).round(3).sort_values("r_con_frio_manana", ascending=False)

In [15]:
columnas_agua = [
    "Agua Planta (Hl)", "Agua Elaboracion (Hl)", "Agua Bodega (Hl)", "Agua Cocina (Hl)",
    "Agua Dilucion (Hl)", "Agua Envasado (Hl)", "Agua Servicios (Hl)", "Planta de agua (Hl)",
    "Produccion (Hl)", "Red L1 y L2", "FC L1 y L2", "FC Lavadora L2", "Red L3", "FC Lavadora L3",
    "Red Paste L4", "FC Barriles", "Agua Paste L3", "Agua Lavadora L3", "Agua Cond Evaporativos",
    "Agua Calderas", "Agua Efluentes", "Agua CO2",
]

df_agua_horario = normalizar_horas(horario_unificado("Consolidado Agua", columnas_agua))
df_agua_diario = df_agua_horario[df_agua_horario["HORA"] == "23:59"]

print("Días con < 24 lecturas:", (df_agua_horario.groupby("DIA").size() < 24).sum())
print("\nControles de calidad:")
print(controles_calidad(df_agua_horario, columnas_agua).to_string())
print("\nCorrelación con Frio:")
print(correlacion_con_frio(df_agua_diario, columnas_agua).to_string())

Filas '00:00' intrusas eliminadas: 1
Horas mal etiquetadas corregidas: 24
Días con < 24 lecturas: 8

Controles de calidad:
                        nulos_%  dias_sin_reinicio  horas_que_bajan  negativos  >=1e6  p99.9_incremento
columna                                                                                                
Agua Planta (Hl)            0.0                  0              105         24      0            3052.0
Agua Elaboracion (Hl)       0.0                  0                5          0      0            1565.2
Agua Bodega (Hl)            0.0                  0                2          0      0             958.4
Agua Cocina (Hl)            0.0                  0                3          0      0             979.5
Agua Dilucion (Hl)          0.0                  0                0          0      0             391.3
Agua Envasado (Hl)          0.0                  0                6          0      0            1582.5
Agua Servicios (Hl)         0.0              

Las 24 horas mal etiquetadas corresponden a un solo dia, el 2021-07-21 en la columna HORA no tiene horas, tiene fechas de 1900 (1900-01-...), un error de formato de Excel. Como el día tiene sus 25 lecturas, normalizar_horas las reetiquetó por posición. Este día coincide con uno de los 5 días con el cierre de Frío inflado (salto de 17.689 kW a las 19:00), lo que refuerza que ese día tuvo problemas de registro.

tambien tenemos variables que no se comportan como acumulador:

 Agua Servicios (Hl) contiene 3.439 horas que bajan y 454 negativos. En un día normal sube bien, pero tantas bajadas sugieren que no es un medidor, sino una columna calculada. Descartada

Agua Paste L3 y Agua Lavadora L3: miles de bajadas y además un 30 % de nulos. Descartada

Agua Planta (Hl) tiene 105 bajadas y 24 negativos: pocos para 30.000 horas, pero lo vamos a tener en cuenta ya que usamos solo el cierre, pero se controlara que el cierre sea razonable

FC Lavadora L2, el cierre vale 0 en el 83% de los dias, su p99.9 de incremento horario es de 16664 Hl (más de 100 veces el de FC Lavadora L3, 140 Hl) y su correlación con el Frío de mañana es de 0,03. Descartada

En cuanto a las correlaciones, vemos que Agua Planta (Hl) r = 0,70 con el frío de mañana, la mejor hasta ahora. Le siguen planta de agua con r=0,69 y produccion con r=0,66 lo que sugiere que posiblemente esten correlacionada entre si. 

una variable interesante era Agua Cond Evaporativos. Los condensadores evaporativos son parte del sistema de refrigeración: es donde se disipa el calor que el frío le saca a la cerveza, evaporando agua. O sea, esa columna mide directamente cuánto trabaja el sistema de frío. Sorprendentemente no fue la mejor ya que su r=0.58, pero hay que tener en cuenta que Pearson solo mide relación lineal: una relación no lineal más fuerte se podría evaluar con Spearman. Posibles razones: no existe en el archivo 2020_2022 (30 % de nulos) y registra incrementos muy chicos (~70 Hl/h), por lo que podría medir solo una parte de los condensadores.

Variables candidatas (sin días sin reinicio, con pocas o ninguna bajada y sin valores imposibles):
| Variable | r con Frío de mañana | Observación |
|---|---|---|
| Agua Planta (Hl) | 0,70 | |
| Planta de agua (Hl) | 0,69 | Posible redundancia con Agua Planta |
| Produccion (Hl) | 0,66 | Posible redundancia con Agua Planta |
| Agua Cond Evaporativos | 0,58 | 30 % nulos (no existe en 2020_2022) |
| Agua Envasado (Hl) | 0,54 | |
| Agua Elaboracion (Hl) | 0,52 | |
| Red L3 | 0,47 | |
| Agua Dilucion (Hl) | 0,45 | |
| Agua Cocina (Hl) | 0,44 | |
| Agua Bodega (Hl) | 0,44 | |
| Agua Efluentes | 0,43 | 30 % nulos |
| Red Paste L4 | 0,41 | |
| Agua Calderas | 0,38 | 30 % nulos |
| Red L1 y L2 | 0,29 | |
| FC L1 y L2 | 0,29 | |
| FC Lavadora L3 | 0,29 | |
| FC Barriles | 0,24 | |
| Agua CO2 | 0,18 | 30 % nulos |
